In [ ]:
import os
os.chdir('./stat_csv')
os.getcwd()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm

In [ ]:
df = pd.read_csv("Harman.8.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df = df.set_index("rownames")
df.head()

In [ ]:
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity, calculate_kmo

chi_square_value, p_value = calculate_bartlett_sphericity(df)
kmo_all, kmo_model = calculate_kmo(df)

print("Bartlett’s test p-value:", p_value)
print("KMO overall:", kmo_model)

# Bartlett p < 0.05 → correlations exist 
# KMO > 0.6 → FA is appropriate 

In [ ]:
# Determine number of factors (Eigenvalues)
from factor_analyzer import FactorAnalyzer
import matplotlib.pyplot as plt

fa = FactorAnalyzer(rotation=None)
fa.fit(df)

eigenvalues, _ = fa.get_eigenvalues()

# Scree plot
plt.figure(figsize=(6,4))
plt.plot(range(1, len(eigenvalues)+1), eigenvalues, marker='o')
plt.axhline(y=1, color='red', linestyle='--')
plt.xlabel("Factor")
plt.ylabel("Eigenvalue")
plt.title("Scree Plot")
plt.show()

#Kaiser criterion → eigenvalue > 1
#Scree “elbow” → typical solution ≈ 3–5 factors

In [ ]:
# Run Factor Analysis: Least Squares (default) (with rotation)
selected_n_factors = ???   # can be adjusted
fa = FactorAnalyzer(n_factors=selected_n_factors, method="minres", rotation='varimax')  # other options for method are 'principal' and 'ml'
fa.fit(df)

factor_columns = []
for i in range(1,selected_n_factors+1):
    factor_columns.append("Factor"+str(i))

loadings = pd.DataFrame(
    fa.loadings_,
    index=df.columns,
    columns=factor_columns
)

print(loadings.round(2))

In [ ]:
# Interpret factor loadings
plt.figure(figsize=(8,10))
sns.heatmap(loadings, annot=True, cmap="coolwarm", center=0)
plt.title("Factor Loadings (Varimax Rotation)")
plt.show()

# High loading (>|0.4|) → variable belongs to factor
# Factors represent latent abilities (e.g. verbal, spatial, memory)

In [ ]:
# Communalities & variance explained
communalities = pd.Series(fa.get_communalities(), index=df.columns)
variance = pd.DataFrame(
    fa.get_factor_variance(),
    index=["SS Loadings", "Proportion Var", "Cumulative Var"],
    columns=factor_columns
)

print("Communalities:")
print(communalities.round(2))

print("\nVariance explained:")
print(variance.round(3))

# Communality → how much variance of a variable is explained
# Cumulative variance → how much total variance factors explain

In [ ]:
# Compute Factor Scores

# Get factor scores for each observation
factor_scores = fa.transform(df)  # shape: (n_samples, n_factors)

factor_scores_df = pd.DataFrame(factor_scores, columns=factor_columns)
print(factor_scores_df.head())